In [ ]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))

import time
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0)
chip.add_compiler("../code/")

# 1.set操作

In [ ]:
pos = (0,256,0,256)
chip.write_point3(*pos,write_voltage=3,tg=1.6,pulse_width=1e-6,set_device=True)

In [ ]:
pos = (0,256,0,256)
voltage_base = chip.read_point3(*pos,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point3(*pos,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond = chip.voltage_to_cond(voltage=voltage-voltage_base)
plot_cond(cond,title="cond",vmin=0,vmax=1500)

In [ ]:
for i in range(20):
    pos = (0,256,0,256)
    
    voltage_base = chip.read_point3(*pos,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    voltage = chip.read_point3(*pos,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    cond = chip.voltage_to_cond(voltage=voltage-voltage_base)
    plot_cond(cond,title=f"cond-write_voltage={(2+i*0.2):.2f}",vmin=0,vmax=1500)


    need_set = cond<200
    print(np.sum(need_set))

    chip.write_point2(crossbar=need_set,write_voltage=2+i*0.2,tg=1.6,pulse_width=1e-6,set_device=True)

# 2.reset操作

In [ ]:
pos = (0,256,0,256)
chip.write_point3(*pos,write_voltage=3,tg=5,pulse_width=10e-6,set_device=False)

In [ ]:
pos = (0,256,0,256)
voltage_base = chip.read_point3(*pos,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point3(*pos,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond = chip.voltage_to_cond(voltage=voltage-voltage_base)
plot_cond(cond,title="cond",vmin=0,vmax=1500)

In [ ]:
pos = (0,256,0,256)
for i in range(20):
    voltage_base = chip.read_point3(*pos,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    voltage = chip.read_point3(*pos,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    cond = chip.voltage_to_cond(voltage=voltage-voltage_base)
    plot_cond(cond,title=f"cond-write_voltage={(2+i*0.2):.2f}",vmin=0,vmax=1500)
    need_reset = cond>1200
    print(np.sum(need_reset))

    chip.write_point2(crossbar=need_reset,write_voltage=2+i*0.2,tg=5,pulse_width=1e-6,set_device=True)

In [ ]:
pos = (30,130,20,120)
# pos = (0,256,0,256)
voltage_base = chip.read_point3(*pos,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point3(*pos,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)

In [ ]:
cond = chip.voltage_to_cond(voltage=voltage-voltage_base)-700
plot_cond(cond,vmin=-500,vmax=500)

interval = 5
bin_edges = np.linspace(-1000, 1000, int(2000/interval)+1)  
data = cond.flatten()
counts, bin_edges, _ = plt.hist(data, bins=bin_edges, color='blue', alpha=0.7, edgecolor='None')

# 3.Forming器件

In [ ]:
def Forming(write_times,write_voltage,start_tg,delta_tg,threshold,set_pulse_width,read_type=2,sub_base=False):
    need_read = np.ones((256,256),dtype=bool)
    voltage = np.zeros((256,256))
    cond_sub_base = None
    for i in range(write_times):
        print(f"write_time = {i}")
        tg = start_tg+i*delta_tg
        if read_type == 2:
            if sub_base:
                voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
            voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        elif read_type == 3:
            pos = (0,256,0,256)
            if sub_base:
                voltage_base = chip.read_point3(*pos,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
            voltage = chip.read_point3(*pos,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
        if sub_base:
            cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        else:
            cond_sub_base = chip.voltage_to_cond(voltage)



        condition_set = cond_sub_base<threshold
        need_read = condition_set

        plot_cond(cond_sub_base,title=f"tg={tg:.2f}-needForming={np.sum(condition_set)}",vmax=1200)
        # if i==0:
        #     chip.write_point2(crossbar=condition_set,write_voltage=5,tg=2.5,pulse_width=1e-6,set_device=False)
        # set的点
        chip.write_point2(crossbar=condition_set,write_voltage=write_voltage,tg=tg,pulse_width=set_pulse_width,set_device=True)

In [ ]:
interval = 1
bin_edges = np.linspace(0, 1500, int(1500 / interval) + 1)

pos = (0,256,0,256)
voltage_base = chip.read_point3(*pos,read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point3(*pos,read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)


In [ ]:

plot_cond(cond_sub_base,vmax=1500)
data = cond_sub_base.flatten()
counts, bin_edges, _ = plt.hist(data, bins=bin_edges, alpha=0.7, edgecolor='none')

In [ ]:
chip.ps.set_time_out(100)
# for i in range(2):
Forming(write_times=301,write_voltage=5,start_tg=1,delta_tg=0.01,threshold=400,set_pulse_width=10e-3,read_type=2)